#  Preprocessing Pipeline

## 1. Overview

This is a simple implementation of a preprocessing pipeline for pedestrian detection using the MOT20 dataset.

The pipeline aims to improve pedestrian detection in crowded scenes through image tiling and adaptive contrast enhancement.

The implementation uses a 2×2 tiling strategy with 20% overlap and applies CLAHE conditionally based on image brightness. YOLOv8 is then used to detect pedestrians in each processed tile.

Processing time is a key consideration in this project, as the system aims to support real-time crowd monitoring and detection. Therefore, the preprocessing pipeline is intentionally kept simple and lightweight to minimize computational overhead while maintaining reliable detection performance.

The main objective is to achieve a balance between detection accuracy and processing efficiency, avoiding unnecessary image enhancement operations that could increase processing time without providing significant improvements in detection performance.

## 2. Preprocessing Pipeline

**Step 1 — Image Loading**

Read the original frames from the MOT20 dataset in chronological order.

**Step 2 — Image Tiling**

Divide each frame into four overlapping tiles (2×2 grid) with 20% overlap to improve the detection of small pedestrians in crowded areas.

**Step 3 — Brightness Analysis**

Calculate the average pixel brightness of each tile. This analysis is performed once every 5,000 frames per sequence to determine whether contrast enhancement is required.

**Step 4 — Contrast Enhancement Method (CLAHE)**

CLAHE (Contrast Limited Adaptive Histogram Equalization) is used as a **contrast enhancement method** to improve visibility in dark and low-contrast regions.

CLAHE is applied only when the average tile brightness is below the predefined threshold. The enhancement is performed on the L channel of the LAB color space to better preserve color information.

**Step 5 — Pedestrian Detection**

Apply YOLOv8 to each tile independently to detect pedestrians.

**Step 6 — Coordinate Transformation**

Convert the detected bounding boxes from local tile coordinates to their corresponding positions in the original frame.

**Step 7 — Duplicate Detection Removal**

Apply Global NMS (Non-Maximum Suppression) and IoS (Intersection over Smaller Area) to remove overlapping or duplicate detections.

**Step 8 — Performance Evaluation**

Evaluate the detection results using Precision, Recall, F1-Score, and processing time.

## 3. Parameter Descriptions

| Parameter | Value | Description |
|---|---|---|
| `MODEL_PATH` | yolov8m.pt | Specifies the YOLO model weights used for pedestrian detection. |
| `SEQUENCES` | MOT20-01, 02, 03 | Defines the dataset sequences used in the experiment. |
| `IMG_SIZE` | 1280 | Specifies the input image size used internally by YOLO. |
| `CONF` | 0.20 | Minimum confidence threshold for accepting detected objects. |
| `YOLO_IOU` | 0.70 | IoU threshold used by YOLO's internal NMS. |
| `MAX_DET` | 1000 | Maximum number of detections allowed per image. |
| `OVERLAP` | 0.20 | Defines the overlap ratio between adjacent image tiles. |
| `BORDER_MARGIN` | 5 | Removes detections close to internal tile boundaries to reduce incomplete bounding boxes. |
| `CLAHE_BRIGHTNESS_THRESHOLD` | 100 | Brightness threshold used to decide whether contrast enhancement is required. |
| `CLAHE_CLIP_LIMIT` | 2.0 | Controls the contrast enhancement strength and limits excessive amplification. |
| `CLAHE_SETTINGS` | 3×3 | Defines the local grid size used by CLAHE. |
| `CLAHE_BLOCK_SIZE` | 5000 | Number of frames processed before recalculating the CLAHE activation decisions for each sequence. |
| `NMS_IOU` | 0.50 | IoU threshold used to identify duplicate bounding boxes after merging tile detections. |
| `IOS_THRESHOLD` | 0.75 | IoS threshold used to identify additional overlapping detections. |
| `CENTER_THRESHOLD` | 0.22 | Maximum normalized center-distance threshold used in additional duplicate detection checks. |
| `SMALL_AREA_RATIO` | 0.75 | Area-ratio threshold used to distinguish smaller overlapping boxes. |
| `MATCH_IOU` | 0.50 | Minimum IoU required to match predictions with ground-truth bounding boxes during evaluation. |
| `PRINT_EVERY` | 100 | Controls how frequently processing progress is displayed. |

## Notes
- The preprocessing configuration is evaluated to study its effect on pedestrian detection accuracy and computational efficiency.
- MOT20-05 is excluded because it is reserved for tune parameters of the preprocessing in validation set.

## 5. Expected Outcome

The expected outcome is to improve pedestrian detection in dense crowds while maintaining reasonable computational performance.

The effectiveness of the preprocessing pipeline is assessed using detection accuracy metrics and average processing time per frame.

In [1]:
import os, json, time, shutil, zipfile, subprocess
import numpy as np
import pandas as pd

from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [3]:
# MOT20: process ALL frames chronologically; one CLAHE decision per 5000 frames PER SEQUENCE; YOLO on tiles only.
# MOT20-05 excluded (reserved for validation). No filter only enhanceme method(CLAHE).
# Designed to run as one Google Colab cell.

import time
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import torch
from ultralytics import YOLO
from scipy.optimize import linear_sum_assignment
from IPython.display import display

# ------------------------ SETTINGS ------------------------
DATASET_PATH = Path('/content/drive/MyDrive/Crowd_AI_Project/datasets/MOT20/train')
MODEL_PATH = 'yolov8m.pt'       # Set to trained best.pt if applicable
SEQUENCES = ['MOT20-01', 'MOT20-02', 'MOT20-03'] #train
CLAHE_BLOCK_SIZE = 5000        # Recheck ONCE per 5000 consecutive frames within each sequence
PRINT_EVERY = 100             # progress every 100 processed frames

IMG_SIZE = 1280
CONF = 0.20
YOLO_IOU = 0.70
MAX_DET = 1000

OVERLAP = 0.20
BORDER_MARGIN = 5
NMS_IOU = 0.50
IOS_THRESHOLD = 0.75
CENTER_THRESHOLD = 0.22
SMALL_AREA_RATIO = 0.75
MATCH_IOU = 0.50

CLAHE_BRIGHTNESS_THRESHOLD = 100.0
CLAHE_CLIP_LIMIT = 2.0
CLAHE_SETTINGS = {
    'MOT20-01': (3, 3),
    'MOT20-02': (3, 3),
    'MOT20-03': (3, 3),
}  # (3,3) is the initial CLAHE grid choice for all three sequences

model = YOLO(MODEL_PATH)


def sync_gpu():
    if torch.cuda.is_available():
        torch.cuda.synchronize()


# ------------------- PROCESS EVERY FRAME IN ORDER -------------------
# Images are processed by ascending frame number for each sequence.
def build_frame_plan():
    plan = {}
    for sequence in SEQUENCES:
        image_dir = DATASET_PATH / sequence / 'img1'
        images = sorted(image_dir.glob('*.jpg'), key=lambda p: int(p.stem))
        gt_path = DATASET_PATH / sequence / 'gt' / 'gt.txt'
        if not images:
            raise FileNotFoundError(f'No images found in {image_dir}')
        if not gt_path.exists():
            raise FileNotFoundError(f'Ground truth missing: {gt_path}')
        plan[sequence] = images
    return plan


# ------------------- CLAHE POLICY: DECIDE ONCE PER BLOCK -------------------
# Compute one brightness score per tile on the FIRST frame in each block.
# Cache four True/False decisions, reuse for all remaining frames in that block.
# There is NO threshold calculation in process_tile(), and NO per-frame check.
def decide_clahe_for_block(first_frame):
    decisions = []
    brightness_values = []
    for tile, _ in create_tiles(first_frame):
        gray = cv2.cvtColor(tile, cv2.COLOR_BGR2GRAY)
        mean_brightness = float(cv2.mean(gray)[0])
        brightness_values.append(mean_brightness)
        decisions.append(mean_brightness < CLAHE_BRIGHTNESS_THRESHOLD)
    return tuple(decisions), tuple(brightness_values)


# ------------------- FIXED CLAHE PER TILE -------------------
def process_tile(tile, sequence, apply_clahe):
    if not apply_clahe:
        return tile, False
    lab = cv2.cvtColor(tile, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(
        clipLimit=CLAHE_CLIP_LIMIT,
        tileGridSize=CLAHE_SETTINGS[sequence]
    )
    enhanced_l = clahe.apply(l)
    out = cv2.cvtColor(cv2.merge((enhanced_l, a, b)), cv2.COLOR_LAB2BGR)
    return out, True


def create_tiles(image):
    h, w = image.shape[:2]
    mid_x, mid_y = w // 2, h // 2
    pad_x = int((w / 2) * OVERLAP / 2)
    pad_y = int((h / 2) * OVERLAP / 2)
    regions = [
        (0, 0, mid_x + pad_x, mid_y + pad_y),
        (mid_x - pad_x, 0, w, mid_y + pad_y),
        (0, mid_y - pad_y, mid_x + pad_x, h),
        (mid_x - pad_x, mid_y - pad_y, w, h),
    ]
    return [(image[y1:y2, x1:x2].copy(), (x1, y1, x2, y2))
            for x1, y1, x2, y2 in regions]


def detect_people(tile):
    result = model.predict(
        source=tile, imgsz=IMG_SIZE, conf=CONF,
        iou=YOLO_IOU, classes=[0], max_det=MAX_DET, verbose=False
    )[0]
    if len(result.boxes) == 0:
        return np.empty((0, 5), dtype=np.float32)
    xyxy = result.boxes.xyxy.cpu().numpy()
    conf = result.boxes.conf.cpu().numpy()
    return np.column_stack((xyxy, conf)).astype(np.float32)


def remove_border_boxes(detections, region, w, h):
    if len(detections) == 0:
        return detections
    rx1, ry1, rx2, ry2 = region
    tw, th = rx2-rx1, ry2-ry1
    border = (
        ((rx1 > 0) & (detections[:, 0] <= BORDER_MARGIN)) |
        ((ry1 > 0) & (detections[:, 1] <= BORDER_MARGIN)) |
        ((rx2 < w) & (detections[:, 2] >= tw-BORDER_MARGIN)) |
        ((ry2 < h) & (detections[:, 3] >= th-BORDER_MARGIN))
    )
    return detections[~border]


def is_duplicate(candidate, existing):
    # Vectorized comparison, with same IoU+IoS+center rules as earlier tests.
    if len(existing) == 0:
        return False
    b = existing[:, :4]
    a = candidate[:4]
    x1 = np.maximum(a[0], b[:, 0])
    y1 = np.maximum(a[1], b[:, 1])
    x2 = np.minimum(a[2], b[:, 2])
    y2 = np.minimum(a[3], b[:, 3])
    inter = np.maximum(0, x2-x1) * np.maximum(0, y2-y1)
    area_a = max(0, a[2]-a[0]) * max(0, a[3]-a[1])
    area_b = np.maximum(0, b[:, 2]-b[:, 0]) * np.maximum(0, b[:, 3]-b[:, 1])
    iou = inter / np.maximum(area_a + area_b - inter, 1e-6)
    if np.any(iou >= NMS_IOU):
        return True
    ios = inter / np.maximum(np.minimum(area_a, area_b), 1e-6)
    ratio = np.minimum(area_a, area_b) / np.maximum(np.maximum(area_a, area_b), 1e-6)
    center_ax = (a[0]+a[2])/2
    center_ay = (a[1]+a[3])/2
    center_bx = (b[:, 0]+b[:, 2])/2
    center_by = (b[:, 1]+b[:, 3])/2
    larger_w = np.where(area_a >= area_b, a[2]-a[0], b[:, 2]-b[:, 0])
    larger_h = np.where(area_a >= area_b, a[3]-a[1], b[:, 3]-b[:, 1])
    duplicate = (
        (ios >= IOS_THRESHOLD) & (ratio <= SMALL_AREA_RATIO) &
        (np.abs(center_ax-center_bx) / np.maximum(larger_w, 1) <= CENTER_THRESHOLD) &
        (np.abs(center_ay-center_by) / np.maximum(larger_h, 1) <= CENTER_THRESHOLD)
    )
    return bool(np.any(duplicate))


def remove_duplicates(detections):
    if len(detections) == 0:
        return detections
    kept = []
    for index in np.argsort(detections[:, 4])[::-1]:
        candidate = detections[index]
        existing = np.asarray(kept, dtype=np.float32).reshape(-1, 5)
        if not is_duplicate(candidate, existing):
            kept.append(candidate)
    return np.asarray(kept, dtype=np.float32).reshape(-1, 5)


# ------------------- DETECTION + TIME PER FRAME -------------------
def run_frame(image, sequence, clahe_decisions):
    h, w = image.shape[:2]
    parts = []
    clahe_tiles = 0
    preprocess_s = 0.0
    yolo_s = 0.0

    sync_gpu()
    frame_start = time.perf_counter()


    for tile_index, (tile, region) in enumerate(create_tiles(image)):
        t0 = time.perf_counter()
        processed, used_clahe = process_tile(
            tile, sequence, clahe_decisions[tile_index]
        )
        preprocess_s += time.perf_counter() - t0
        clahe_tiles += int(used_clahe)

        sync_gpu()
        t0 = time.perf_counter()
        boxes = detect_people(processed)
        sync_gpu()
        yolo_s += time.perf_counter() - t0

        boxes = remove_border_boxes(boxes, region, w, h)
        if len(boxes):
            x1, y1, x2, y2 = region
            boxes[:, [0, 2]] += x1
            boxes[:, [1, 3]] += y1
            boxes[:, [0, 2]] = np.clip(boxes[:, [0, 2]], 0, w)
            boxes[:, [1, 3]] = np.clip(boxes[:, [1, 3]], 0, h)
            parts.append(boxes)

    t0 = time.perf_counter()
    combined = np.vstack(parts) if parts else np.empty((0, 5), dtype=np.float32)
    final = remove_duplicates(combined)
    merge_s = time.perf_counter() - t0
    sync_gpu()
    total_s = time.perf_counter() - frame_start

    return final, {
        'CLAHE Tiles': clahe_tiles,
        'Preprocessing (s)': preprocess_s,
        'YOLO (s)': yolo_s,
        'Merge (s)': merge_s,
        'Total (s)': total_s,
    }


# ------------------- GT: READ ONCE PER SEQUENCE -------------------
def load_gt_by_frame(path):
    raw = pd.read_csv(path, header=None, usecols=[0, 2, 3, 4, 5, 6, 7])
    raw.columns = ['frame', 'x', 'y', 'w', 'h', 'valid', 'class_id']
    raw = raw[(raw['valid'] == 1) & (raw['class_id'] == 1)]
    by_frame = {}
    for frame, group in raw.groupby('frame'):
        xywh = group[['x', 'y', 'w', 'h']].to_numpy(dtype=np.float32)
        boxes = np.column_stack((xywh[:, 0], xywh[:, 1],
                                 xywh[:, 0]+xywh[:, 2], xywh[:, 1]+xywh[:, 3]))
        by_frame[int(frame)] = boxes.astype(np.float32)
    return by_frame


def evaluate(detections, gt_boxes):
    pred = detections[:, :4]
    if len(gt_boxes) == 0 or len(pred) == 0:
        tp = 0
    else:
        gt = gt_boxes[:, None, :]
        pr = pred[None, :, :]
        ix1 = np.maximum(gt[..., 0], pr[..., 0])
        iy1 = np.maximum(gt[..., 1], pr[..., 1])
        ix2 = np.minimum(gt[..., 2], pr[..., 2])
        iy2 = np.minimum(gt[..., 3], pr[..., 3])
        inter = np.maximum(0, ix2-ix1)*np.maximum(0, iy2-iy1)
        a = np.maximum(0, gt[..., 2]-gt[..., 0])*np.maximum(0, gt[..., 3]-gt[..., 1])
        b = np.maximum(0, pr[..., 2]-pr[..., 0])*np.maximum(0, pr[..., 3]-pr[..., 1])
        iou = inter/np.maximum(a+b-inter, 1e-6)
        valid = iou >= MATCH_IOU
        scores = np.where(valid, 1000.0+iou, 0)
        rows, cols = linear_sum_assignment(scores, maximize=True)
        tp = int(valid[rows, cols].sum())
    return tp, len(pred)-tp, len(gt_boxes)-tp


# ------------------- WARM-UP (NOT TIMED) -------------------
plan = build_frame_plan()
print('Frame plan:', {sequence: len(paths) for sequence, paths in plan.items()})
print('Total frames to process (NO sampling):', sum(map(len, plan.values())))
print('CLAHE: available for ALL three sequences, (3,3) grid')
print(f'CLAHE decision: ONCE per {CLAHE_BLOCK_SIZE} frames PER sequence')
print(f'CLAHE threshold: first-frame tile brightness < {CLAHE_BRIGHTNESS_THRESHOLD}')
print('MOT20-05: EXCLUDED | Guided: OFF | Full-image YOLO: OFF')

first_sequence = next(iter(plan))
first_image = cv2.imread(str(plan[first_sequence][0]))
if first_image is None:
    raise ValueError('Warm-up image cannot be read.')
_ = detect_people(create_tiles(first_image)[0][0])
sync_gpu()

# ------------------- PROCESS ALL AVAILABLE FRAMES -------------------
results = []
block_decisions_log = []
processed_count = 0
planned_total = sum(map(len, plan.values()))
wall_start = time.perf_counter()

for sequence, paths in plan.items():
    gt_by_frame = load_gt_by_frame(DATASET_PATH / sequence / 'gt' / 'gt.txt')
    print(f'\nProcessing {sequence}: ALL {len(paths)} frames (in order)')

    # Policy starts fresh for each sequence. Recheck every 5000 processed
    # frames within that same sequence, never on intervening frames.
    clahe_decisions = (False, False, False, False)
    processed_in_sequence = 0
    for path in paths:
        image = cv2.imread(str(path))
        if image is None:
            print('Unreadable:', path)
            continue

        decision_s = 0.0
        if processed_in_sequence % CLAHE_BLOCK_SIZE == 0:
            t0 = time.perf_counter()
            clahe_decisions, scores = decide_clahe_for_block(image)
            decision_s = time.perf_counter() - t0
            block_number = processed_in_sequence // CLAHE_BLOCK_SIZE + 1
            block_decisions_log.append({
                'Dataset': sequence,
                'Block': block_number,
                'Decision Frame': int(path.stem),
                'Brightness by Tile': tuple(round(v, 2) for v in scores),
                'Apply CLAHE by Tile': clahe_decisions,
            })
            print(f'  Block {block_number} brightness: '
                  f'{[round(v, 2) for v in scores]}')
            print(f'  CLAHE enabled (tiles 1..4): {clahe_decisions}')

        frame_number = int(path.stem)
        detections, stats = run_frame(image, sequence, clahe_decisions)
        # Include the one-time policy decision in the first frame of its block.
        stats['Decision (s)'] = decision_s
        stats['Total (s)'] += decision_s

        gt = gt_by_frame.get(frame_number, np.empty((0, 4), dtype=np.float32))
        tp, fp, fn = evaluate(detections, gt)
        precision = tp/(tp+fp) if tp+fp else 0.0
        recall = tp/(tp+fn) if tp+fn else 0.0
        f1 = 2*precision*recall/(precision+recall) if precision+recall else 0.0

        results.append({
            'Dataset': sequence, 'Frame': frame_number,
            'GT': len(gt), 'Detections': len(detections),
            'TP': tp, 'FP': fp, 'FN': fn,
            'Precision': precision, 'Recall': recall, 'F1': f1,
            **stats,
        })
        processed_count += 1
        processed_in_sequence += 1
        if processed_count % PRINT_EVERY == 0:
            recent = results[-PRINT_EVERY:]
            avg_s = np.mean([r['Total (s)'] for r in recent])
            print(f'{processed_count}/{planned_total} frames | '
                  f'{sequence} | recent mean: {avg_s:.3f}s/frame | '
                  f'CLAHE on latest frame: {stats["CLAHE Tiles"]}/4')

print('\n========== CLAHE BLOCK DECISIONS ==========')
display(pd.DataFrame(block_decisions_log))

# ------------------- FINAL TABLES (NO PER-FRAME IMAGE DISPLAY) -------------------
df_no_guided_5000 = pd.DataFrame(results)
if df_no_guided_5000.empty:
    raise RuntimeError('No frames were successfully processed.')

print('\n========== PER-DATASET SUMMARY ==========')
summary_rows = []
for sequence, part in df_no_guided_5000.groupby('Dataset', sort=False):
    tp, fp, fn = int(part.TP.sum()), int(part.FP.sum()), int(part.FN.sum())
    p = tp/(tp+fp) if tp+fp else 0
    r = tp/(tp+fn) if tp+fn else 0
    f1 = 2*p*r/(p+r) if p+r else 0
    mean_s = part['Total (s)'].mean()
    summary_rows.append({
        'Dataset': sequence, 'Frames': len(part),
        'GT': int(part.GT.sum()), 'TP': tp, 'FP': fp, 'FN': fn,
        'Precision': p, 'Recall': r, 'F1': f1,
        'CLAHE Tiles Used': int(part['CLAHE Tiles'].sum()),
        'CLAHE Use (%)': 100*part['CLAHE Tiles'].sum()/(4*len(part)),
        'Mean Preprocessing (s)': part['Preprocessing (s)'].mean(),
        'Total Decision Time (s)': part['Decision (s)'].sum(),
        'Mean YOLO (s)': part['YOLO (s)'].mean(),
        'Mean Total (s)': mean_s,
        'FPS': 1/mean_s if mean_s else 0,
    })
summary = pd.DataFrame(summary_rows)
display(summary.round(3))

all_tp, all_fp, all_fn = map(int, (df_no_guided_5000.TP.sum(),
                                  df_no_guided_5000.FP.sum(),
                                  df_no_guided_5000.FN.sum()))
all_p = all_tp/(all_tp+all_fp) if all_tp+all_fp else 0
all_r = all_tp/(all_tp+all_fn) if all_tp+all_fn else 0
all_f1 = 2*all_p*all_r/(all_p+all_r) if all_p+all_r else 0
avg = df_no_guided_5000['Total (s)'].mean()
print('\n========== TOTAL ==========')
print(f'Frames evaluated: {len(df_no_guided_5000)}')
print(f'Precision: {all_p:.2%} | Recall: {all_r:.2%} | F1: {all_f1:.2%}')
print(f'Average processing: {avg:.3f} s/frame | Approx. processing FPS: {1/avg:.2f}')
print(f'Total pipeline compute time: {df_no_guided_5000["Total (s)"].sum():.1f} s')
print(f'Elapsed wall time (includes image read, GT matching, printing): {time.perf_counter()-wall_start:.1f} s')

# Per-frame results: df_no_guided_5000 | Per-sequence/block decisions: block_decisions_log
# No output files are saved to Drive. All input frames are processed sequentially.


Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
Frame plan: {'MOT20-01': 429, 'MOT20-02': 2782, 'MOT20-03': 2405}
Total frames to process (NO sampling): 5616
CLAHE: available for ALL three sequences, (3,3) grid
CLAHE decision: ONCE per 5000 frames PER sequence
CLAHE threshold: first-frame tile brightness < 100.0
MOT20-05: EXCLUDED | Guided: OFF | Full-image YOLO: OFF

Processing MOT20-01: ALL 429 frames (in order)
  Block 1 brightness: [117.21, 118.83, 82.1, 110.01]
  CLAHE enabled (tiles 1..4): (False, False, True, False)
100/5616 frames | MOT20-01 | recent mean: 0.371s/frame | CLAHE on latest frame: 1/4


KeyboardInterrupt: 